# HyperDailyTsvDatabase Demo

这个 notebook 演示新的本地优先数据库入口：初始化数据库、获取并落盘日线、读取分钟线和实时行情。

In [9]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import free_market_data.providers as providers_module
import free_market_data.providers.hyper_provider as hyper_provider_module
import free_market_data.daily_db as daily_db_module
importlib.reload(hyper_provider_module)
importlib.reload(providers_module)
importlib.reload(daily_db_module)

HyperProvider = hyper_provider_module.HyperProvider
DEFAULT_PROVIDER_CLASSES = providers_module.DEFAULT_PROVIDER_CLASSES
from free_market_data.daily_db import HyperDailyTsvDatabase
from free_market_data.symbols import normalize_stock_code

In [10]:
provider_order = ('akshare', 'baostock', 'yahoo', 'tencent', 'xueqiu', 'sina', 'sohu')
provider_map = {name: cls() for name, cls in DEFAULT_PROVIDER_CLASSES.items()}
hyper_provider = HyperProvider(provider_map=provider_map, provider_order=provider_order)

db = HyperDailyTsvDatabase.initialize(
    root_dir=ROOT / 'data' / 'hyper_daily_db',
    hyper_provider=hyper_provider,
    providers=provider_order,
)

db

In [4]:
stock_code = '000001'
start_date = '2024-01-02'
end_date = '2024-03-29'

daily = db.get_daily(
    stock_codes=stock_code,
    start_date=start_date,
    end_date=end_date,
    refresh=True,
)

print('input stock_code:', stock_code)
print('normalized stock_code:', daily['stock_code'].iloc[0] if not daily.empty else None)
print('rows:', len(daily))
print('columns:', list(daily.columns))
daily.head()

input stock_code: 000001
normalized stock_code: 000001.SZ
rows: 58
columns: ['date', 'stock_code', 'open', 'close', 'high', 'low', 'volume', 'amount', 'source', 'qfq_factor', 'hfq_factor', 'price_source', 'updated_at', 'adj_close', 'change', 'pct_change', 'turnover']


,date,stock_code,open,close,high,low,volume,amount,source,qfq_factor,hfq_factor,price_source,updated_at,adj_close,change,pct_change,turnover
0,2024-01-02,000001.SZ,9.39,9.21,9.42,9.21,1158366.0,107574.22,"tencent,yahoo,sohu",0.830293,176.425299,tencent,2026-05-11 17:21:54,7.992280,-0.18,NaN,NaN
1,2024-01-03,000001.SZ,9.19,9.20,9.22,9.15,733610.0,67367.36,"tencent,yahoo,sohu",0.830109,176.453370,tencent,2026-05-11 17:21:54,7.983603,-0.01,NaN,NaN
2,2024-01-04,000001.SZ,9.19,9.11,9.19,9.08,864194.0,78747.01,"tencent,yahoo,sohu",0.828430,176.709440,tencent,2026-05-11 17:21:54,7.905502,-0.09,NaN,NaN
3,2024-01-05,000001.SZ,9.10,9.27,9.44,9.07,1991622.0,185265.97,"tencent,yahoo,sohu",0.831392,176.257713,tencent,2026-05-11 17:21:54,8.044347,0.16,NaN,NaN
4,2024-01-08,000001.SZ,9.23,9.15,9.30,9.11,1121156.0,102900.66,"tencent,yahoo,sohu",0.829180,176.595082,tencent,2026-05-11 17:21:54,7.940213,-0.12,NaN,NaN


In [4]:
normalized_stock_code = normalize_stock_code(stock_code)
cache_path = ROOT / 'data' / 'hyper_daily_db' / 'daily' / f'{normalized_stock_code}.tsv'
print('cache exists:', cache_path.exists())
cache_path

cache exists: True


WindowsPath('d:/2026_claude/StockFrame/data/hyper_daily_db/daily/000001.SZ.tsv')

## Schema From Cached Data

下面这一节不是写死的 schema，而是直接根据数据库里已经存在的日线缓存生成字段介绍。

In [ ]:
schema_intro = db.describe_schema()
print('schema fields:', len(schema_intro))
schema_intro

In [5]:
close_matrix = db.get_daily(
    stock_codes=stock_code,
    start_date=start_date,
    end_date=end_date,
    fields='close',
    refresh=False,
    return_format='wide',
)

close_matrix.tail()

stock_code,000001.SZ
date,
2024-03-25,10.40
2024-03-26,10.60
2024-03-27,10.53
2024-03-28,10.49
2024-03-29,10.52


## Batch Symbols

下面这一节验证同一个数据库对象对股票列表的批量处理能力。

In [8]:
batch_stock_codes = ['000001', '600519']

batch_daily = db.get_daily(
    stock_codes=batch_stock_codes,
    start_date=start_date,
    end_date=end_date,
    refresh=True,
)

batch_minute = db.get_minute(
    stock_codes=batch_stock_codes,
    period='5m',
)

batch_realtime = db.get_realtime(batch_stock_codes)

print('batch daily codes:', sorted(batch_daily['stock_code'].dropna().unique().tolist()))
print('batch minute codes:', sorted(batch_minute['stock_code'].dropna().unique().tolist()))
print('batch realtime codes:', sorted(batch_realtime['stock_code'].dropna().unique().tolist()))

batch_daily.head()

batch daily codes: ['000001.SZ', '600519.SH']
batch minute codes: ['000001.SZ', '600519.SH']
batch realtime codes: ['000001.SZ', '600519.SH']


,date,stock_code,open,close,high,low,volume,amount,source,qfq_factor,hfq_factor,price_source,updated_at,adj_close,change,pct_change,turnover
0,2024-01-02,000001.SZ,9.39,9.21,9.42,9.21,1158366.0,107574.22,"tencent,yahoo,sohu",0.830293,176.425299,tencent,2026-05-11 17:10:54,7.992281,-0.18,NaN,NaN
1,2024-01-02,600519.SH,1715.00,1685.01,1718.19,1678.10,32156.0,544008.25,"tencent,yahoo,sohu",0.936905,6.249298,tencent,2026-05-11 17:10:57,1567.384277,-40.99,NaN,NaN
2,2024-01-03,000001.SZ,9.19,9.20,9.22,9.15,733610.0,67367.36,"tencent,yahoo,sohu",0.830109,176.453370,tencent,2026-05-11 17:10:54,7.983602,-0.01,NaN,NaN
3,2024-01-03,600519.SH,1681.11,1694.00,1695.22,1676.33,20229.0,341140.06,"tencent,yahoo,sohu",0.937240,6.245999,tencent,2026-05-11 17:10:57,1575.746582,8.99,NaN,NaN
4,2024-01-04,000001.SZ,9.19,9.11,9.19,9.08,864194.0,78747.01,"tencent,yahoo,sohu",0.828430,176.709440,tencent,2026-05-11 17:10:54,7.905502,-0.09,NaN,NaN


## Minute And Realtime

分钟线和实时行情通过同一个数据库对象拿，但当前不会写入本地 TSV。

In [ ]:
minute = db.get_minute(stock_code=stock_code, period='5m')
print('minute rows:', len(minute))
minute.head()

In [11]:
realtime = db.get_realtime([stock_code], fields=['price', 'volume', 'timestamp'])
print('realtime rows:', len(realtime))
print('realtime columns:', list(realtime.columns))
realtime

realtime rows: 1
realtime columns: ['stock_code', 'timestamp', 'price', 'volume']


,stock_code,timestamp,price,volume
0,000001.SZ,2026-05-11 17:27:11.890222,11.28,93186521.0


In [12]:
realtime_check = db.get_realtime([stock_code], fields=['price', 'volume', 'timestamp'])
print('realtime check columns:', list(realtime_check.columns))
print(realtime_check.to_string(index=False))

realtime check columns: ['stock_code', 'timestamp', 'price', 'volume']
stock_code                  timestamp  price     volume
 000001.SZ 2026-05-11 17:27:24.895603  11.28 93186521.0
